# BIAN 310 · Module 4 | Sales Performance Visualization Challenge
### Instructor demonstration + student practice · 1 notebook · 2 modes

**The assignment itself is unchanged.** With `DEMO_MODE` checked, this notebook uses the **nine records supplied in Assignment 4.1**. Uncheck it to use an independent **240-person fictional practice dataset**. The notebook will create a cleaned Excel workbook, five KPI calculations, three charts, a visual dashboard, a Word writing guide, and a ZIP file.

**Important:** Completed visualizations are learning aids. Students must explain their chart choice, inspect the findings, and write their *own* 150–300-word business recommendation. Historic bonuses are not a defensible sole basis for future bonuses.


In [ ]:
# STEP 1 — Set your details and choose the dataset.
# In Google Colab, click the small form/checkbox controls beside this cell.
DEMO_MODE = True #@param {type:"boolean"}
STUDENT_NAME = "" #@param {type:"string"}
ASSIGNMENT = "Module 4: Assignment 4.1 — Sales Performance Visualization Challenge" #@param {type:"string"}

if not STUDENT_NAME.strip():
    STUDENT_NAME = input("Enter your first and last name: ").strip()
if not ASSIGNMENT.strip():
    ASSIGNMENT = input("Enter your assignment name: ").strip()
if not STUDENT_NAME:
    raise ValueError("Please enter your name before proceeding.")
print(f"Student: {STUDENT_NAME} | Assignment: {ASSIGNMENT}")
print("Selected data:", "Original 9-person instructor demo" if DEMO_MODE else "240-person student practice")


In [ ]:
# STEP 2 — Ensure required packages are installed (prevents xlsxwriter errors).
import sys, subprocess, importlib.util
required = {"pandas":"pandas", "numpy":"numpy", "matplotlib":"matplotlib",
            "xlsxwriter":"XlsxWriter", "docx":"python-docx"}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from datetime import datetime
from IPython.display import display, Image
from zipfile import ZipFile, ZIP_DEFLATED
import random, re, io, csv
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
print("Dependencies ready. Running in Colab:", IN_COLAB)


In [ ]:
# STEP 3 — Load the original assignment data OR the 240-person practice data.
# The practice dataset is built into the notebook, so no separate upload is required.
FIELDS = ['Salesperson','Total Sales ($)','Avg. Previous Bonus ($)',
          'Customer Accounts','Years with Company']
DEMO_ROWS = [
 ('Smith, Michael',325000.78,12499.3452,124,14),
 ('Yu, Joe',13678.21,239.9434,9,7),
 ('Reeves, Bill',452359.19,21987.2462,175,21),
 ('Hamilton, Joshua',87423.91,7642.9011,28,3),
 ('Harper, Derek',87654.21,1250.1393,21,4),
 ('Quinn, Dorothy',234091.39,14567.9833,48,9),
 ('Graves, Lorrie',379401.94,27981.4432,121,12),
 ('Sun, Yi',31733.59,672.9111,7,1),
 ('Thompson, Nicole',127845.22,13322.9713,17,3)]
FIRST = ['Alex','Avery','Blair','Casey','Cameron','Drew','Emery','Finley',
 'Harper','Jordan','Kai','Kendall','Logan','Morgan','Parker','Quinn',
 'Reese','Riley','Rowan','Sage','Skyler','Taylor','Terry','Dakota']
LAST = ['Anderson','Baker','Carter','Diaz','Ellis','Foster','Garcia','Hayes',
 'Irwin','Jones','Kim','Lewis','Mitchell','Nelson','Owens','Patel','Reed',
 'Santos','Turner','Vega','Walker','Young','Zimmerman','Brooks']
def generate_student_data():
    rng = random.Random(31041)
    rows=[]
    for j in range(240):
        name=f'{LAST[j//10]}, {FIRST[j%24]} {j//24+1:02d}'
        years=rng.randint(1,24)
        accounts=rng.randint(8,150)
        sales=max(11000,accounts*rng.uniform(1300,2900)+
                  years*rng.uniform(1000,4900)+rng.gauss(0,38000))
        if j%37==0: sales*=1.36
        bonus=max(0,sales*rng.uniform(.026,.074)+rng.gauss(0,4700))
        rows.append((name,round(sales,2),round(bonus,4),accounts,years))
    return rows

source = pd.DataFrame(DEMO_ROWS if DEMO_MODE else generate_student_data(),columns=FIELDS)
source['Dataset'] = 'Original assignment (9)' if DEMO_MODE else 'Fictional student practice (240)'
print("Loaded",len(source),"records")
display(source.head(10))


In [ ]:
# STEP 4 — Data audit and professional cleaning (raw source is preserved).
raw = source.copy(deep=True)
clean = source[FIELDS].copy()
clean['Salesperson'] = clean['Salesperson'].astype('string').str.strip()
numeric_fields = FIELDS[1:]
for field in numeric_fields:
    clean[field] = pd.to_numeric(clean[field], errors='coerce')
issues = pd.DataFrame([
    {'Check':'Records loaded','Finding':len(clean)},
    {'Check':'Missing cells','Finding':int(clean[FIELDS].isna().sum().sum())},
    {'Check':'Duplicate salesperson names','Finding':int(clean['Salesperson'].duplicated().sum())},
    {'Check':'Negative sales/bonus','Finding':int((clean[FIELDS[1:3]]<0).sum().sum())},
    {'Check':'Negative accounts/tenure','Finding':int((clean[FIELDS[3:]]<0).sum().sum())}])
if clean[FIELDS].isna().any().any() or (clean[numeric_fields]<0).any().any():
    raise ValueError('Please review the data-quality report before using these records.')
clean = clean.sort_values('Total Sales ($)',ascending=False,kind='stable').reset_index(drop=True)
print('Sorted by sales, highest first. All original records remain preserved.')
display(issues)
display(clean.head(10))


In [ ]:
# STEP 5 — Calculate the exact five requested KPIs and supporting relationships.
kpis = {
 'Total Sales':float(clean['Total Sales ($)'].sum()),
 'Average Sales per Salesperson':float(clean['Total Sales ($)'].mean()),
 'Highest Individual Sales':float(clean['Total Sales ($)'].max()),
 'Average Previous Bonus':float(clean['Avg. Previous Bonus ($)'].mean()),
 'Total Customer Accounts':int(clean['Customer Accounts'].sum())}
relationship = {
 'Sales vs previous bonus':clean['Total Sales ($)'].corr(clean['Avg. Previous Bonus ($)']),
 'Sales vs years with company':clean['Total Sales ($)'].corr(clean['Years with Company']),
 'Sales vs customer accounts':clean['Total Sales ($)'].corr(clean['Customer Accounts'])}
print('FIVE ASSIGNMENT KPIs')
for key,value in kpis.items(): print(f'{key}: {value:,.0f}' if key=='Total Customer Accounts' else f'{key}: ${value:,.2f}')
print('\nExploratory correlations (association, not causation):')
for key,value in relationship.items():print(f'  {key}: r={value:.2f}')
print('Highest:',clean.iloc[0]['Salesperson'],'| Lowest:',clean.iloc[-1]['Salesperson'])


In [ ]:
# STEP 6 — Generate three assignment charts, readable in BOTH data sizes.
out_dir = Path('/content/BIAN310_Module4_Outputs') if IN_COLAB else Path.cwd()/'BIAN310_Module4_Outputs'
out_dir.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'figure.dpi':135,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
navy, blue, teal, amber = '#15334f','#2375aa','#14a69a','#e5a640'

def save_chart(filename,fig):
    fig.savefig(out_dir/filename,bbox_inches='tight',facecolor='white')
    plt.close(fig)

# Chart 1: Top 20 ranked employees for readability; full 240 remain in Excel.
top = clean.head(20).iloc[::-1]
fig,ax=plt.subplots(figsize=(11,7 if len(top)>9 else 5.5))
ax.barh(top['Salesperson'],top['Total Sales ($)']/1000,color=blue)
ax.set(xlabel='Total sales ($ thousands)',title='Chart 1 | Total Sales by Salesperson')
ax.grid(axis='x',alpha=.18)
if len(clean)>20:ax.text(0,-.13,'Top 20 displayed; complete ranking is in the Excel workbook.',transform=ax.transAxes,fontsize=9)
save_chart('01_sales_ranking.png',fig)

# Chart 2: Scatter avoids misleading same-scale bars for sales and bonuses.
fig,ax=plt.subplots(figsize=(10,5.5))
ax.scatter(clean['Total Sales ($)']/1000,clean['Avg. Previous Bonus ($)']/1000,
           c=teal,s=54,alpha=.76,edgecolors='white',linewidths=.5)
ax.set(xlabel='Total sales ($ thousands)',ylabel='Previous average bonus ($ thousands)',
       title='Chart 2 | Sales vs. Previous Bonus')
ax.grid(alpha=.18)
save_chart('02_sales_vs_bonus.png',fig)

# Chart 3: Accounts vs sales is an independent relationship; bubble size shows tenure.
fig,ax=plt.subplots(figsize=(10,5.5))
sc=ax.scatter(clean['Customer Accounts'],clean['Total Sales ($)']/1000,
              s=28+clean['Years with Company']*7,c=clean['Years with Company'],
              cmap='viridis',alpha=.7,edgecolors='white',linewidths=.4)
ax.set(xlabel='Customer accounts',ylabel='Total sales ($ thousands)',
       title='Chart 3 | Customer Accounts vs. Sales (size/color = tenure)')
fig.colorbar(sc,ax=ax,label='Years with company')
ax.grid(alpha=.18)
save_chart('03_accounts_vs_sales.png',fig)
print('Three charts saved:',*[p.name for p in sorted(out_dir.glob('0*.png'))],sep='\n  ')


In [ ]:
# STEP 7 — Build and SHOW the one-page executive dashboard.
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(16,10),facecolor='#f4f7fb')
gs=GridSpec(3,6,figure=fig,height_ratios=[.8,3,3],hspace=.53,wspace=.65)
fig.text(.06,.96,'SALES PERFORMANCE | MANAGER DASHBOARD',fontsize=20,fontweight='bold',color=navy)
fig.text(.06,.925,('ORIGINAL ASSIGNMENT DEMO' if DEMO_MODE else '240-PERSON PRACTICE')+'  •  Training data — not a compensation decision',fontsize=10,color='#566777')
labels=list(kpis)
for j,(name,val) in enumerate(kpis.items()):
    x=.06+j*.185
    fig.text(x,.865,name.upper(),fontsize=9,color='#46677f',fontweight='bold')
    fig.text(x,.831,f'{val:,.0f}' if j==4 else f'${val:,.0f}',fontsize=17,color=navy,fontweight='bold')
a=fig.add_subplot(gs[1,0:3]); top10=clean.head(10).iloc[::-1]
a.barh(top10['Salesperson'],top10['Total Sales ($)']/1000,color=blue)
a.set_title('Sales leaders · top 10',loc='left',fontweight='bold');a.set_xlabel('Sales ($000s)')
b=fig.add_subplot(gs[1,3:6]);b.scatter(clean['Total Sales ($)']/1000,clean['Avg. Previous Bonus ($)']/1000,color=teal,alpha=.65)
b.set_title('Sales vs previous bonuses',loc='left',fontweight='bold');b.set_xlabel('Sales ($000s)');b.set_ylabel('Previous bonus ($000s)')
c=fig.add_subplot(gs[2,0:3]);c.scatter(clean['Customer Accounts'],clean['Total Sales ($)']/1000,c=amber,alpha=.68)
c.set_title('Accounts vs sales',loc='left',fontweight='bold');c.set_xlabel('Customer accounts');c.set_ylabel('Sales ($000s)')
d=fig.add_subplot(gs[2,3:6]);d.axis('off')
lines=[f'Highest sales: {clean.iloc[0]["Salesperson"]}',f'Lowest sales: {clean.iloc[-1]["Salesperson"]}',
       f'Sales / bonus correlation: {relationship["Sales vs previous bonus"]:.2f}',
       f'Sales / tenure correlation: {relationship["Sales vs years with company"]:.2f}',
       f'Sales / accounts correlation: {relationship["Sales vs customer accounts"]:.2f}',
       'Review margin, targets, retention and account quality',
       'before making ANY bonus recommendation.']
d.text(.02,.94,'WHAT TO DISCUSS',fontsize=14,fontweight='bold',color=navy)
for i,line in enumerate(lines):d.text(.02,.82-i*.105,'• '+line,fontsize=11,color=navy)
fig.savefig(out_dir/'Sales_Executive_Dashboard.png',bbox_inches='tight',facecolor=fig.get_facecolor())
plt.close(fig)
display(Image(filename=str(out_dir/'Sales_Executive_Dashboard.png'),width=1000))


In [ ]:
# STEP 8 — Produce a fully formatted Excel workbook with KPIs, charts and source.
# NOTE: XlsxWriter was checked/installed in Step 2.
excel_path=out_dir/'Sales_Performance_Analysis.xlsx'
with pd.ExcelWriter(excel_path,engine='xlsxwriter') as writer:
    clean.to_excel(writer,sheet_name='Cleaned Sales Data',index=False,startrow=2)
    raw.to_excel(writer,sheet_name='Original Source',index=False,startrow=2)
    issues.to_excel(writer,sheet_name='Quality Checks',index=False,startrow=2)
    book=writer.book
    header=book.add_format({'bold':True,'font_color':'white','bg_color':navy,'border':0,'text_wrap':True})
    currency=book.add_format({'num_format':'$#,##0.00'})
    integer=book.add_format({'num_format':'#,##0'})
    title=book.add_format({'bold':True,'font_color':navy,'font_size':17})
    note=book.add_format({'font_color':'#567186','italic':True,'text_wrap':True})
    valuefmt=book.add_format({'bold':True,'font_size':16,'font_color':'#087b70','num_format':'$#,##0.00'})
    countfmt=book.add_format({'bold':True,'font_size':16,'font_color':'#087b70','num_format':'#,##0'})
    for sheetname,frame in [('Cleaned Sales Data',clean),('Original Source',raw),('Quality Checks',issues)]:
        sh=writer.sheets[sheetname]
        sh.write(0,0,sheetname+' | '+('DEMO' if DEMO_MODE else 'PRACTICE'),title)
        sh.freeze_panes(3,1); sh.set_column('A:A',29);sh.set_column('B:C',24,currency);sh.set_column('D:E',22,integer)
        if sheetname=='Original Source':sh.set_column('F:F',35)
        if sheetname=='Quality Checks':sh.set_column('A:A',37);sh.set_column('B:B',24)
        for c,name in enumerate(frame.columns):sh.write(2,c,name,header)
        if len(frame):sh.autofilter(2,0,len(frame)+2,len(frame.columns)-1)
    dash=book.add_worksheet('Dashboard');dash.set_tab_color(teal)
    dash.hide_gridlines(2);dash.set_column('A:A',3);dash.set_column('B:F',19);dash.set_column('G:O',16)
    dash.merge_range('B2:M3','SALES PERFORMANCE | MANAGER DASHBOARD',title)
    dash.merge_range('B4:M4','Original 9-person assignment demonstration' if DEMO_MODE else '240-person fictional practice dataset',note)
    kpi_cells=['B6','E6','H6','K6','N6']
    for i,(label,number) in enumerate(kpis.items()):
        cell=kpi_cells[i];col=int(re.match(r'[A-Z]+',cell).group().replace('B','2').replace('E','5').replace('H','8').replace('K','11').replace('N','14')) if False else [1,4,7,10,13][i]
        dash.write(5,col,label,header)
        # Excel formulas, not fixed values: adjust automatically after data edits.
        rlast=len(clean)+3
        formula=["SUM('Cleaned Sales Data'!B4:B%d)"%rlast,
                 "AVERAGE('Cleaned Sales Data'!B4:B%d)"%rlast,
                 "MAX('Cleaned Sales Data'!B4:B%d)"%rlast,
                 "AVERAGE('Cleaned Sales Data'!C4:C%d)"%rlast,
                 "SUM('Cleaned Sales Data'!D4:D%d)"%rlast][i]
        dash.write_formula(6,col,'='+formula,countfmt if i==4 else valuefmt,number)
    dash.insert_image('B10',str(out_dir/'01_sales_ranking.png'),{'x_scale':.58 if len(clean)>9 else .63,'y_scale':.56 if len(clean)>9 else .63})
    dash.insert_image('H10',str(out_dir/'02_sales_vs_bonus.png'),{'x_scale':.60,'y_scale':.60})
    dash.insert_image('B36',str(out_dir/'03_accounts_vs_sales.png'),{'x_scale':.65,'y_scale':.65})
    dash.write('H39','Chart 3 choice: accounts vs sales.',header)
    dash.merge_range('H41:O44','Why this chart? It examines whether employees managing more customer accounts also generate more sales. Bubble size and color show tenure. This is exploratory—not evidence that accounts cause sales.',note)
    dash.merge_range('H47:O50','Bonus review: compare profitability, goal attainment, account complexity, customer retention, territory differences, and company policy before final decisions.',note)
    dash.freeze_panes(5,0)
    # Put dashboard first for the manager.
    dash.activate();dash.set_first_sheet()
print('Saved:',excel_path.name)


In [ ]:
# STEP 9 — Create the Word document: evidence, questions and writing structure.
doc=Document();sec=doc.sections[0];sec.top_margin=Inches(.7);sec.bottom_margin=Inches(.7)
doc.styles['Normal'].font.name='Aptos';doc.styles['Normal'].font.size=Pt(10)
doc.add_heading('BIAN 310 | Sales Visualization: Student Analysis Guide',0)
doc.add_paragraph('Student: '+STUDENT_NAME+'    |    Assignment: '+ASSIGNMENT)
doc.add_paragraph('Dataset: '+('9-person original assignment DEMO' if DEMO_MODE else '240-person fictional STUDENT PRACTICE'))
doc.add_heading('Assignment 4.1 checklist (requirements unchanged)',1)
for line in [
 'Clean and format data in Excel; keep unrounded underlying values.',
 'Include three clearly titled and labeled charts.',
 'Present all five calculated KPIs.',
 'Explain why Chart 3 (customer accounts versus sales) was selected.',
 'Write your own 150–300-word analysis and a conditional managerial recommendation.',
 'Submit the Excel workbook and the written analysis. Review figures before submission.']:
    doc.add_paragraph(line,style='List Bullet')
doc.add_heading('Evidence to check against the Excel workbook',1)
table=doc.add_table(rows=1,cols=2);table.style='Light Shading Accent 1'
table.rows[0].cells[0].text='Measure';table.rows[0].cells[1].text='Computed value'
for k,v in kpis.items():
    row=table.add_row().cells;row[0].text=k;row[1].text=f'{v:,.0f}' if k=='Total Customer Accounts' else f'${v:,.2f}'
doc.add_paragraph(f'Highest sales: {clean.iloc[0]["Salesperson"]} | Lowest sales: {clean.iloc[-1]["Salesperson"]}')
doc.add_heading('Interpretation prompts: respond to all seven',1)
qs=['Who generated the highest sales?','Who generated the lowest sales?',
    'Does the previous bonus appear related to total sales?',
    'Do years with the company appear to influence performance?',
    'Do customer account counts appear to influence sales?',
    'Whose bonus should management review, and why?',
    'What additional evidence is needed before a FINAL bonus decision?']
for q in qs:doc.add_paragraph(q,style='List Number')
doc.add_heading('Draft your own 150–300 words below',1)
for label in ['Opening finding (40–60 words):','Charts and relationships (60–100 words):',
              'Conditional bonus recommendation (50–90 words):','Missing evidence and limitations (30–50 words):']:
    doc.add_paragraph(label,style='Heading 2')
    doc.add_paragraph('[Write your own analysis here.]')
doc.add_heading('Good business analysis cautions',1)
doc.add_paragraph('Correlation does not establish causation. Higher sales may reflect more accounts, larger territories, product mix, or higher quotas. Previous bonuses are historical decisions, not proof of deserved future bonuses. Check profit margin, attainment versus target, account retention, data quality, eligibility and compensation policy. This is a classroom exercise.')
doc.add_heading('Academic integrity',1)
doc.add_paragraph('Use this generated file as a guide, not a submitted analysis. Check all calculations, customize your Chart 3 explanation, acknowledge automated tools when required, and write your own recommendations.')
word_path=out_dir/'Student_Analysis_and_Submission_Guide.docx'
doc.save(word_path)
print('Saved:',word_path.name)


In [ ]:
# STEP 10 — Package deliverables and AUTO-DOWNLOAD a single ZIP in Colab.
csv_path=out_dir/('Original_9_Person_Demo.csv' if DEMO_MODE else 'Student_Practice_240_People.csv')
source[FIELDS].to_csv(csv_path,index=False)
kpi_path=out_dir/'Five_KPIs.csv'
pd.DataFrame([{'KPI':name,'Value':value} for name,value in kpis.items()]).to_csv(kpi_path,index=False)
issues.to_csv(out_dir/'Data_Quality_Checks.csv',index=False)
notes=(f'Student: {STUDENT_NAME}\nAssignment: {ASSIGNMENT}\n'
       f'Mode: {"DEMO / original assignment" if DEMO_MODE else "STUDENT / fictional practice"}\n'
       'Open the Excel dashboard; review 3 PNG charts; write your own 150-300-word recommendation in the Word guide.\n'
       'No bonus decisions are automated. All figures are for training.\n')
(out_dir/'START_HERE.txt').write_text(notes,encoding='utf-8')
safe_name=re.sub(r'[^a-zA-Z0-9_-]+','_',STUDENT_NAME).strip('_')[:36] or 'Student'
zip_path=out_dir/f'BIAN310_Module4_{safe_name}_{"DEMO" if DEMO_MODE else "STUDENT"}.zip'
with ZipFile(zip_path,'w',ZIP_DEFLATED) as z:
    include = [csv_path,kpi_path,out_dir/'Data_Quality_Checks.csv',
      out_dir/'START_HERE.txt',out_dir/'Sales_Executive_Dashboard.png',
      out_dir/'Sales_Performance_Analysis.xlsx',out_dir/'Student_Analysis_and_Submission_Guide.docx',
      out_dir/'01_sales_ranking.png',out_dir/'02_sales_vs_bonus.png',out_dir/'03_accounts_vs_sales.png']
    for file in include:
        z.write(file,arcname=file.name)
print(f'PACKAGE READY: {zip_path.name} ({zip_path.stat().st_size/1024/1024:.2f} MB)')
print('Files:',*[p.name for p in sorted(out_dir.iterdir()) if p.is_file() and p!=zip_path and not p.name.endswith('.zip')],sep='\n  ')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print('Outside Colab: ZIP available at',zip_path.resolve())
